### Import Libraries

In [ ]:
# Bibliotecas RAPIDS (GPU)
from cuml.preprocessing import StandardScaler
from cuml.pipeline import Pipeline
from cuml.neighbors import KNeighborsClassifier
from cuml.svm import SVC
from cuml.ensemble import RandomForestClassifier
import xgboost as xgb
from cuml.metrics import accuracy_score
from cuml.metrics import confusion_matrix

# Bibliotecas Scikit-learn (CPU)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler as SklearnStandardScaler 
from sklearn.metrics import precision_recall_fscore_support

# Bibliotecas de Utilidades
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
import cupy as cp
import numpy as np
import pandas as pd # <--- GARANTIR PANDAS
import seaborn as sns
import matplotlib.pyplot as plt
import gc

# Bibliotecas TensorFlow
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# Configuração de Memória GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

### Audio Preprocessing

In [ ]:
# 1. Carregar os dados do GTZAN
# Assumindo que o arquivo features_30_sec.csv está no diretório correto.
# Ajuste o caminho conforme sua organização de pastas.
GTZAN_CSV_PATH = '../gtzan_dataset/features_3_sec.csv' 

data = pd.read_csv(GTZAN_CSV_PATH)

print(f"Dataset carregado. Shape original: {data.shape}")

# 2. Separar Features (X) e Target (y)
# O GTZAN csv padrão tem 'filename', 'length' e 'label'. O resto são features.
# A monografia menciona 57 atributos (pág 29). O CSV padrão costuma ter mean e var para várias features.
# Vamos remover as colunas não-features.
drop_cols = ['filename', 'length', 'label']
X_all_pd = data.drop(columns=drop_cols)
y_all_labels_pd = data['label']

print(f"Total de faixas 'GTZAN': {X_all_pd.shape[0]}")
print(f"Total de features: {X_all_pd.shape[1]}")

# 3. EXTRAIR GRUPOS (CRÍTICO PARA 3 SEGUNDOS)
# Para evitar que pedaços da mesma música caiam em treino e teste simultaneamente,
# usamos o nome do arquivo como "ID do Grupo".
# No CSV de 3s, as 10 partes da música 'blues.00000.wav' têm o mesmo filename.
groups_np = data['filename'].to_numpy()

print(f"Grupos definidos. Exemplo: {groups_np[:5]}")

# 4. Codificar os Gêneros (Labels)
label_encoder = LabelEncoder()
y_all_encoded_np = label_encoder.fit_transform(y_all_labels_pd).astype(np.int32)

# 5. Converter X para Numpy
X_all_np = X_all_pd.to_numpy()

# Verificar classes
print("Classes encontradas:", label_encoder.classes_)

In [ ]:
# Converta para float32 para economizar RAM e VRAM e compatibilidade com cuML
X_data = X_all_np.astype(np.float32)
y_data = y_all_encoded_np.astype(np.int32)

# Definir número de classes (Deve ser 10 para o GTZAN)
num_classes = len(np.unique(y_data))
print(f"Número de classes identificadas: {num_classes}")

### Treino dos Modelos

In [ ]:
def build_reference_mlp(input_shape, num_classes, params):
    # Extrai os parâmetros otimizados ou usa defaults se não existirem
    n_layers = params.get('n_layers', 4)
    n_units = params.get('units', 512)
    dropout_rate = params.get('dropout', 0.4)
    l2_reg = params.get('l2_reg', 0.002)
    lr = params.get('lr', 0.0005)
    
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))

    # Loop para criar as camadas dinamicamente
    for i in range(n_layers):
        # A cada camada, podemos reduzir o número de neurónios (ex: 256 -> 128 -> 64)
        # ou mantê-lo constante. Aqui, reduzo pela metade a cada camada para criar um funil.
        current_units = max(32, n_units // (2 ** i)) 
        
        model.add(layers.Dense(
            current_units, 
            kernel_initializer='he_normal',
            kernel_regularizer=regularizers.l2(l2_reg)
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.ELU())
        model.add(layers.Dropout(dropout_rate))
    
    # Camada de Saída
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr) 
    
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
def objective(trial, model_name, X, y, groups):
    # Split interno para validação rápida
    inner_cv = GroupKFold(n_splits=3)
    
    # --- ESPAÇO DE BUSCA DOS HIPERPARÂMETROS ---
    # (Mantenha todo o bloco de 'if model_name == ...' idêntico ao original)
    if model_name == "KNN (GPU)":
        n_neighbors = trial.suggest_int("n_neighbors", 3, 40)
        model = KNeighborsClassifier(n_neighbors=n_neighbors)
    elif model_name == "SVM (GPU)":
        C = trial.suggest_float("C", 0.1, 100.0, log=True)
        gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
        kernel = trial.suggest_categorical("kernel", ["rbf", "poly"]) 
        model = SVC(C=C, gamma=gamma, kernel=kernel)
    elif model_name == "Random Forest (GPU)":
        n_estimators = trial.suggest_int("n_estimators", 50, 300)
        max_depth = trial.suggest_int("max_depth", 5, 30)
        max_features = trial.suggest_float("max_features", 0.1, 1.0)
        model = RandomForestClassifier(n_estimators=n_estimators, 
                                       max_depth=max_depth, 
                                       max_features=max_features)
    elif model_name == "XGBoost (GPU)":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "tree_method": "hist",
            "device": "cuda",
            "verbosity": 0,
            "objective": 'multi:softmax',
            "num_class": num_classes
        }
        model = xgb.XGBClassifier(**params)
    elif model_name == "MLP (Keras)":
        # (Copiar o bloco MLP do original, é idêntico)
        params = {
            'n_layers': trial.suggest_int("n_layers", 1, 4),
            'units': trial.suggest_int("units", 128, 512, step=64),
            'dropout': trial.suggest_float("dropout", 0.1, 0.5),
            'l2_reg': trial.suggest_float("l2_reg", 1e-5, 1e-2, log=True),
            'lr': trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        }
        scores = []
        # ALTERADO: split(X, y) sem groups
        for train_idx, val_idx in inner_cv.split(X, y, groups=groups):
            X_t, X_v = X[train_idx], X[val_idx]
            y_t, y_v = y[train_idx], y[val_idx]

            # === SELEÇÃO DE FEATURES (Mantenha idêntico ao original) ===
            fs_model = xgb.XGBClassifier(
                n_estimators=50, max_depth=3, tree_method='hist', device="cuda", 
                random_state=42, objective='multi:softmax', num_class=num_classes, verbosity=0
            )
            fs_model.fit(X_t, y_t)
            importances = fs_model.feature_importances_
            threshold = np.mean(importances) * 1.25
            select_mask = importances > threshold
            if np.sum(select_mask) < 10:
                top_indices = np.argsort(importances)[-20:]
                select_mask[:] = False
                select_mask[top_indices] = True
            X_t = X_t[:, select_mask]
            X_v = X_v[:, select_mask]
            # ===============================================

            scaler = SklearnStandardScaler()
            X_t = scaler.fit_transform(X_t)
            X_v = scaler.transform(X_v)
            
            k_model = build_reference_mlp(X_t.shape[1], num_classes, params)
            k_model.fit(X_t, y_t, epochs=20, batch_size=64, verbose=0)
            acc = k_model.evaluate(X_v, y_v, verbose=0)[1]
            scores.append(acc)
            
            del k_model
            tf.keras.backend.clear_session()
            gc.collect()
        return np.mean(scores)

    # --- AVALIAÇÃO PADRÃO (RAPIDS/XGBoost) ---
    scores = []
    # ALTERADO: split(X, y) sem groups
    for train_idx, val_idx in inner_cv.split(X, y, groups=groups):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        # === SELEÇÃO DE FEATURES (Idêntico ao anterior) ===
        fs_model = xgb.XGBClassifier(
            n_estimators=50, max_depth=3, tree_method='hist', device="cuda", 
            random_state=42, objective='multi:softmax', num_class=num_classes, verbosity=0
        )
        fs_model.fit(X_train_fold, y_train_fold)
        importances = fs_model.feature_importances_
        threshold = np.mean(importances) * 1.25
        select_mask = importances > threshold
        if np.sum(select_mask) < 10:
            top_indices = np.argsort(importances)[-20:]
            select_mask[:] = False
            select_mask[top_indices] = True
        X_train_fold = X_train_fold[:, select_mask]
        X_val_fold = X_val_fold[:, select_mask]
        # ===============================================

        if model_name in ["KNN (GPU)", "SVM (GPU)"]:
            scaler = StandardScaler()
            X_train_fold = cp.array(X_train_fold)
            X_val_fold = cp.array(X_val_fold)
            y_train_fold = cp.array(y_train_fold)
            y_val_fold = cp.array(y_val_fold)
            X_train_fold = scaler.fit_transform(X_train_fold)
            X_val_fold = scaler.transform(X_val_fold)
        elif model_name == "XGBoost (GPU)":
             pass
        else: # Random Forest
             X_train_fold = cp.array(X_train_fold).astype(np.float32)
             X_val_fold = cp.array(X_val_fold).astype(np.float32)
             y_train_fold = cp.array(y_train_fold).astype(np.int32)
             y_val_fold = cp.array(y_val_fold).astype(np.int32)

        model.fit(X_train_fold, y_train_fold)
        preds = model.predict(X_val_fold)
        
        if isinstance(preds, cp.ndarray): preds = cp.asnumpy(preds)
        if isinstance(y_val_fold, cp.ndarray): y_val_fold = cp.asnumpy(y_val_fold)
            
        scores.append(accuracy_score(y_val_fold, preds))
        
    return np.mean(scores)

In [ ]:
# Lista de modelos a otimizar
target_models = ["KNN (GPU)", "SVM (GPU)", "Random Forest (GPU)", "XGBoost (GPU)", "MLP (Keras)"]
best_models_config = {}

print("--- INICIANDO OTIMIZAÇÃO DE HIPERPARÂMETROS (OPTUNA) ---")

# Usa-se uma amostra dos dados para otimização caso o dataset for muito grande
# Mas como FMA_small é pequeno, pode-se usar tudo.
X_opt = X_all_np
y_opt = y_all_encoded_np
groups_opt = groups_np

for name in target_models:
    print(f"\nOtimizando {name}...")
    
    # Define a função de estudo específica para o modelo atual
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    
    # Lambda para passar os argumentos extras
    func = lambda trial: objective(trial, name, X_opt, y_opt, groups=groups_opt)
    
    # Número de tentativas (trials). 
    # Para testes rápidos use 10-20. Para resultado final use 50-100.
    n_trials = 20 if name != "MLP (Keras)" else 10 
    
    study.optimize(func, n_trials=n_trials)
    
    print(f"Melhores params para {name}: {study.best_params}")
    print(f"Melhor score (CV interno): {study.best_value:.4f}")
    
    best_models_config[name] = study.best_params
    tf.keras.backend.clear_session()
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()

print("\n--- OTIMIZAÇÃO CONCLUÍDA ---")

In [ ]:
# 4. Definir a Estratégia de CV
n_splits = 10
kf = GroupKFold(n_splits=n_splits)

# 5. Criar Pipelines para os Modelos
# Isso garante que o StandardScaler seja "fitado" apenas nos dados
# de treino de cada fold, e depois "transforma" os dados de treino e teste.

# Recriar os modelos finais com os melhores parâmetros encontrados
models = {}

# 1. KNN
p_knn = best_models_config["KNN (GPU)"]
# Nota: precisamos recriar o Pipeline pois o scaler é parte dele
models["KNN (GPU)"] = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=p_knn['n_neighbors']))
])

# 2. SVM
p_svm = best_models_config["SVM (GPU)"]
models["SVM (GPU)"] = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(C=p_svm['C'], gamma=p_svm['gamma'], kernel=p_svm['kernel']))
])

# 3. Random Forest
p_rf = best_models_config["Random Forest (GPU)"]
models["Random Forest (GPU)"] = RandomForestClassifier(
    n_estimators=p_rf['n_estimators'],
    max_depth=p_rf['max_depth'],
    max_features=p_rf['max_features']
)

# 4. XGBoost
p_xgb = best_models_config["XGBoost (GPU)"]
# Adicionamos parâmetros fixos necessários que não foram otimizados
p_xgb.update({
    "tree_method": 'hist', "device": "cuda", 
    "objective": 'multi:softmax', "num_class": num_classes,
    "random_state": 42
})
models["XGBoost (GPU)"] = xgb.XGBClassifier(**p_xgb)

# 5. MLP
# Para o MLP, passaremos o dicionário de config diretamente para ser usado dentro do loop de treino principal
models["MLP (Keras)"] = best_models_config["MLP (Keras)"]

# 6. Treinar de Modelos
# Dicionário para guardar os scores de CADA métrica para cada modelo
# Ex: cv_scores['KNN']['Accuracy'] = [0.5, 0.52, ...]
cv_scores = {}

# Dicionário para guardar as previsões e rótulos de todos os folds,
# para a matriz de confusão final
out_of_fold_preds = {}

print(f"Iniciando treinamento com {n_splits} folds...")
print(f"Shape dos dados: {X_all_np.shape}")

debug_feature_selection_flag = True

# --- LOOP DOS MODELOS ---
for model_name, model in models.items():
    print(f"\nIniciando CV 10-Fold para {model_name}...")
    
    fold_scores_acc = []
    fold_scores_precision = []
    fold_scores_recall = []
    fold_scores_f1 = []
    
    all_y_true = []
    all_y_pred = []

    y_pred_fold = None # Variável para guardar as previsões deste fold

    # --- LOOP DOS K-FOLDS ---
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_all_np, y_all_encoded_np, groups=groups_np)):
        # Dados em Numpy (CPU) inicialmente
        X_train, X_test = X_all_np[train_idx], X_all_np[test_idx]
        y_train, y_test = y_all_encoded_np[train_idx], y_all_encoded_np[test_idx]

        # === SELEÇÃO DE FEATURES COM XGBOOST (GPU) ===
        ## A. Treinar o seletor rápido
        feature_selection_model = xgb.XGBClassifier(
            n_estimators=50,
            max_depth=3,
            tree_method='hist',
            device="cuda", 
            random_state=42,
            objective='multi:softmax',
            num_class=num_classes,
            verbosity=0
        )
        feature_selection_model.fit(X_train, y_train)

        ## B. Calcular Importâncias
        importances = feature_selection_model.feature_importances_

        ## C. Definir o Limiar
        ## Isso é uma heurística comum: "Seja um pouco melhor que a média para ficar"
        threshold = np.mean(importances) * 1.25 # ...altere esse número para alterar o limiar

        ## Seleciona booleanos onde a importância > threshold
        select_mask = importances > threshold

        ## D. Filtrar
        # Fallback: Se selecionar muito poucas (<10), pega as top 20
        if np.sum(select_mask) < 10:
            top_indices = np.argsort(importances)[-20:]
            select_mask[:] = False
            select_mask[top_indices] = True
            
        X_train = X_train[:, select_mask]
        X_test = X_test[:, select_mask]

        # Debug: Mostra quantas features foram selecionadas.
        if debug_feature_selection_flag == True:
            n_selected = np.sum(select_mask)
            print(f"O modelo escolheu automaticamente {n_selected} features relevantes.")

            debug_feature_selection_flag = False
        
        # =============================================
        
        # --- TREINO DOS MODELOS ---
        ## 1. MLP (TensorFlow/Keras)
        if model_name == "MLP (Keras)":
            # Scaling na CPU para o TF
            scaler = SklearnStandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Build & Train
            best_params = model
            keras_model = build_reference_mlp(X_train.shape[1], num_classes, best_params)
            callbacks_list = [
                # Para se o modelo não melhorar por 8 épocas (evita desperdício)
                EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
                
                # Reduz a taxa de aprendizado se o modelo estagnar (O "Pulo do Gato")
                # Se o loss não cair por 3 épocas, divide o LR por 5. 
                # Isso ajuda o modelo a descer o "vale" do mínimo global com passos menores.
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', 
                    factor=0.2, 
                    patience=3, 
                    min_lr=1e-6,
                    verbose=1
                )
            ]

            # Treinamento
            history = keras_model.fit(
                X_train_scaled, y_train,
                epochs=70,
                batch_size=32,
                validation_split=0.1,
                callbacks=callbacks_list,
                verbose=0
            )
            
            # Predict
            y_probs = keras_model.predict(X_test_scaled, verbose=0)
            y_pred_fold = np.argmax(y_probs, axis=1)
            
            # Limpeza Crítica para GPU 6GB
            tf.keras.backend.clear_session()
            del keras_model, X_train_scaled, X_test_scaled
            gc.collect()

        ## 2. Modelos GPU
        elif "(GPU)" in model_name:
            # Converter para CuPy (GPU)
            X_train_cp = cp.array(X_train)
            X_test_cp = cp.array(X_test)
            y_train_cp = cp.array(y_train)
            
            model.fit(X_train_cp, y_train_cp)
            y_pred_cp = model.predict(X_test_cp)
            
            # Trazer de volta para CPU para métricas unificadas depois
            y_pred_fold = cp.asnumpy(y_pred_cp)
            
            # Limpeza
            del X_train_cp, X_test_cp, y_train_cp, y_pred_cp
        
        # Acurácia (cuml accuracy_score aceita numpy e devolve float ou array 0-d)
        acc = accuracy_score(y_test, y_pred_fold)
        fold_scores_acc.append(acc)
        
        # Calculo dos precision_recall_fscore_support
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_test, y_pred_fold, average='macro', zero_division=0)
        
        # Converter métricas scalarem de volta para CPU/float para armazenar na lista
        fold_scores_precision.append(float(macro_p))
        fold_scores_recall.append(float(macro_r))
        fold_scores_f1.append(float(macro_f1))
        
        # Acumular para Matriz Final
        all_y_true.append(y_test)
        all_y_pred.append(y_pred_fold)
    
    # Armazenar resultados do modelo
    cv_scores[model_name] = {
        'Acurácia': np.array(fold_scores_acc),
        'Precisão': np.array(fold_scores_precision),
        'Recall': np.array(fold_scores_recall),
        'F1-Score': np.array(fold_scores_f1)
    }
    
    out_of_fold_preds[model_name] = {
        'y_true': np.concatenate(all_y_true),
        'y_pred': np.concatenate(all_y_pred)
    }
    
    # Limpeza final do modelo
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

print("\n--- Avaliação Concluída ---")

### Análise de Resultados

In [ ]:
for model_name, metrics in cv_scores.items():
    print(f"\n========= {model_name} =========")
    for metric_name, values in metrics.items():
        print(f"{metric_name:15}: Média {values.mean():.4f} | Std {values.std():.4f}")

# Plot Matriz de Confusão
class_names = label_encoder.classes_

for model_name, results in out_of_fold_preds.items():
    cm = confusion_matrix(results['y_true'], results['y_pred'])
    if hasattr(cm, 'get'): cm = cm.get() # Se for cupy, move para numpy
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'GTZAN - {model_name}')
    plt.ylabel('Real')
    plt.xlabel('Previsto')
    plt.show()

# MLP Treining Measurement
if 'history' in locals():
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Gráfico de Acurácia
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo-', label='Acurácia no Treino')     # Linha Azul
    plt.plot(epochs, val_acc, 'ro-', label='Acurácia na Validação') # Linha Vermelha
    plt.title('Acurácia: Treino vs Validação (MLP)')
    plt.xlabel('Épocas')
    plt.ylabel('Acurácia')
    plt.legend()

    # Gráfico de Perda (Loss)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo-', label='Perda no Treino')
    plt.plot(epochs, val_loss, 'ro-', label='Perda na Validação')
    plt.title('Perda: Treino vs Validação (MLP)')
    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()
else:
    print("A variável 'history' não foi encontrada. Certifique-se de ter rodado o treino do MLP.")